In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, linear_layer, rank=2, alpha=1.0):
        super().__init__()
        self.scale = alpha / rank
        self.linear_layer = linear_layer
        if rank <= 0:
            raise ValueError("rank must be > 0")

        self.A = nn.Parameter(torch.randn((rank, self.linear_layer.in_features))) # Initiate gaussian A matrix
        self.B = nn.Parameter(torch.zeros((self.linear_layer.out_features, rank))) # Initiate zeros B matrix

        for param in self.linear_layer.parameters():
            param.requires_grad = False # Freeze original params

    def forward(self, x):
        return self.linear_layer(x) + self.scale * (x @ self.A.T @ self.B.T) # Update weight matrix with sclaled delta

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def apply_lora(model, rank=2, alpha=1.0):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            lora = LoRALinear(module, rank=rank, alpha=alpha)
            parent = model
            parts = name.split('.')
            for p in parts[:-1]:
                parent = getattr(parent, p) # Replacing linear layer with LoRA linear layer
            setattr(parent, parts[-1], lora)
    return model

def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    correct_zeros = 0
    total_zeros = 0

    with torch.no_grad():
        for img, label in dataloader:
            img, label = img.to(device), label.to(device)
            out = model(img)
            preds = out.argmax(dim=1)
            correct += (preds == label).sum().item()
            total += label.size(0)

            zero_mask = (label == 0)
            if zero_mask.sum() > 0: # Count zeros
                correct_zeros += (preds[zero_mask] == label[zero_mask]).sum().item()
                total_zeros += zero_mask.sum().item()

    return correct / total, correct_zeros / total_zeros if total_zeros > 0 else 0

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor()])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

In [ ]:
base_model = SimpleNN().to(device)
optimizer = optim.Adam(base_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

base_model.train()
for epoch in range(5): # Train base model before fine-tuning
    for img, label in train_loader:
        img, label = img.to(device), label.to(device)
        optimizer.zero_grad()
        out = base_model(img)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()

base_acc, base_zero_acc = evaluate(base_model, test_loader, device)

zero_indices = [i for i, (_, label) in enumerate(train_set) if label == 0] # Create dataset with more zeros for fine-tuning
zero_subset = torch.utils.data.Subset(train_set, zero_indices * 5)  # Repeat zeros 5x
zero_loader = DataLoader(zero_subset, batch_size=64, shuffle=True)

model = apply_lora(base_model, rank=2, alpha=4.0)
model.to(device)

lora_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"LoRA tunable parameters: {lora_params:,}")
print(f"Total parameters: {total_params:,}")

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

model.train()
for epoch in range(3): # Fine-tuning on 0s
    for img, label in zero_loader:
        img, label = img.to(device), label.to(device)
        optimizer.zero_grad()
        out = model(img)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()

lora_acc, lora_zero_acc = evaluate(model, test_loader, device)

print(f"Base Model - General Accuracy: {base_acc:.4f}, Zero Accuracy: {base_zero_acc:.4f}")
print(f"LoRA Model - General Accuracy: {lora_acc:.4f}, Zero Accuracy: {lora_zero_acc:.4f}")

LoRA tunable parameters: 2,356
Total parameters: 111,742
Base Model - General Accuracy: 0.9762, Zero Accuracy: 0.9837
LoRA Model - General Accuracy: 0.6996, Zero Accuracy: 1.0000
